# OpenAI API 呼叫方式介紹

Responses API vs legacy Chat Completions

![image.png](attachment:image.png)

## 模組脈絡：API 參數就是「可控性槓桿」

本筆記是 **01-不可控的根源** 的起點。看似單純的 API 介紹，其實每個參數都是你能施加的控制：

- **temperature**：決定輸出的隨機程度——非確定性的源頭（見 [02-sampling-and-uncertainty]）。
- **instructions / input 角色**：意圖注入的第一層（見模組 02 意圖收斂）。
- **token 上限與錯誤碼**：行為邊界與失敗模式。

學完本模組你應理解「**為何 LLM 難以控制**」，後續模組再逐層把它收斂回可控。

In [1]:
# Import necessary libraries
## 設定 OpenAI API Key 變數
from dotenv import load_dotenv
import os

# 載入 .env 文件中的環境變數
load_dotenv()

# 獲取 API 金鑰
openai_api_key = os.getenv('OPENAI_API_KEY')

# 使用 API 金鑰
# print(f'The API key is: {api_key}')


In [ ]:
## 如果使用 colab, 可以使用以下方式設定 API key 
# from google.colab import userdata
# openai_api_key = userdata.get('OPENAI_API_KEY')

### 傳統 HTTP library 格式，透過 payload 和 header 進行 HTTP 請求

大部分的人都是使用 OpenAI 的 notebook 都用 OpenAI 官方出的 https://github.com/openai/openai-python

In [2]:
import requests
import json # 這有兩個方法 dumps (物件轉字串) 跟 loads (字串轉物件)
from pprint import pp # 為了印出來漂亮

## OpenAI Responses API（2026 新專案建議入口）

> **注意**：`/v1/completions` 與舊式 instruct 模型已淘汰；`/v1/chat/completions` 仍可用於簡單聊天或舊系統相容，但新教材與新專案請優先使用 **Responses API**。

In [3]:
# raw HTTP 版：用來理解底層協定；實務請優先用官方 SDK
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

payload = {
    "model": OPENAI_MODEL,
    "temperature": 1,
    "input": [{"role": "user", "content": "頑皮豹的膚色是?"}],
    "max_output_tokens": 100,
}

headers = {
    "Authorization": f"Bearer {openai_api_key}",
    "Content-Type": "application/json",
}

response = requests.post("https://api.openai.com/v1/responses", headers=headers, data=json.dumps(payload), timeout=30)
obj = response.json()
pp(obj)

{'id': 'resp_070e87baf6bc1aff006a3ba5ebbd7c819ba1004636b4cf0207',
 'object': 'response',
 'created_at': 1782293995,
 'status': 'completed',
 'background': False,
 'billing': {'payer': 'developer'},
 'completed_at': 1782293996,
 'error': None,
 'frequency_penalty': 0.0,
 'incomplete_details': None,
 'instructions': None,
 'max_output_tokens': 100,
 'max_tool_calls': None,
 'model': 'gpt-5.4-mini-2026-03-17',
 'moderation': None,
 'output': [{'id': 'msg_070e87baf6bc1aff006a3ba5ec6790819ba499f9414f503938',
             'type': 'message',
             'status': 'completed',
             'content': [{'type': 'output_text',
                          'annotations': [],
                          'logprobs': [],
                          'text': '頑皮豹（Pink Panther）的膚色是**粉紅色**。'}],
             'phase': 'final_answer',
             'role': 'assistant'}],
 'parallel_tool_calls': True,
 'presence_penalty': 0.0,
 'previous_response_id': None,
 'prompt_cache_key': None,
 'prompt_cache_retention': '24

## OpenAI Chat API

在OpenAI的API中，當我們使用Chat模式進行多輪對話時，我們會通過一個名為messages的陣列來傳遞對話記錄。這個陣列中的每一項都代表對話中的一個訊息，而這些訊息依照角色（role）的不同有不同的含義：

* system: 用於設定和調整整個對話的行為，通常作為對話陣列messages的第一個元素。它可以指定對話的某些規則或者參數設定，幫助引導對話的方向或者行為。
* user: 代表使用者的訊息。當我們想要模擬使用者向系統提問或者進行互動時，會使用這個角色。
* assistant: 代表AI助手的回覆。在一次對話完成後，如果想要進行多輪對話，我們需要將上一次AI的回覆作為一部分加入到messages陣列中，以便系統可以理解對話的上下文。

若只進行一輪對話，則messages陣列中可以只包含一個角色為user的訊息，這時候的使用方式與Completion API非常相似，你只需要將整個對話的內容放入該訊息的content中即可。這樣，API就會根據這個單一的輸入訊息來生成回覆。

In [4]:
import requests
import json

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

def get_openai_response_http(messages, model=OPENAI_MODEL, temperature=0, max_output_tokens=300, timeout=30):
    """用 raw HTTP 呼叫 Responses API；保留此版本是為了看懂底層 payload。"""
    payload = {
        "model": model,
        "temperature": temperature,
        "input": messages,
        "max_output_tokens": max_output_tokens,
    }
    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
    }
    response = requests.post("https://api.openai.com/v1/responses", headers=headers, data=json.dumps(payload), timeout=timeout)
    obj = response.json()
    if response.status_code == 200:
        return obj.get("output_text", obj)
    return obj.get("error", {"message": "An unknown error occurred"})

In [5]:
# 現代寫法（2026 production 標準）：openai>=2.0 官方 SDK + Responses API
# 上面的 requests 版是為了看懂底層 HTTP 協定；實務請用 SDK（自帶重試、型別、串流）
from openai import OpenAI

client = OpenAI()  # 自動讀取環境變數 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

def get_openai_completion(messages, model=OPENAI_MODEL, temperature=0, max_output_tokens=300, timeout=30):
    """以 Responses API 取得一般文字輸出。messages 可沿用 role/content 陣列。"""
    try:
        resp = client.responses.create(
            model=model,
            input=messages,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
            timeout=timeout,
        )
        return resp.output_text
    except Exception as e:  # 速率限制 / 內容政策 / 連線錯誤都會走這裡
        return f"[API 錯誤] {type(e).__name__}: {e}"

print(get_openai_completion([{"role": "user", "content": "用一句話介紹 Python"}]))

Python 是一種簡潔易讀、用途廣泛的高階程式語言，常用於網頁開發、資料分析、人工智慧和自動化。


In [6]:
user_message = "什麼是 python? 請用台灣繁體中文" # 可以改任意問題


## array 格式輸入
messages = [
    {
        "role": "user",
        "content": user_message
    }
]

response = get_openai_completion(messages, temperature=0)
pp(response)

('Python 是一種**程式語言**，用來寫電腦程式、網站、資料分析、人工智慧、自動化腳本等。\n'
 '\n'
 '### 簡單來說\n'
 '你可以把 Python 想成是「跟電腦溝通的語言」。  \n'
 '你用 Python 寫出指令，電腦就照著做。\n'
 '\n'
 '### Python 的特色\n'
 '- **語法簡單、好讀**：很適合初學者\n'
 '- **用途廣泛**：可以做網站、資料處理、機器學習、爬蟲、自動化\n'
 '- **社群很大**：有很多現成套件可以用\n'
 '- **跨平台**：Windows、Mac、Linux 都能跑\n'
 '\n'
 '### 例子\n'
 '```python\n'
 'print("你好，Python！")\n'
 '```\n'
 '\n'
 '這段程式會在畫面上顯示：\n'
 '```python\n'
 '你好，Python！\n'
 '```\n'
 '\n'
 '### 常見用途\n'
 '- **網站開發**：例如 Django、Flask\n'
 '- **資料分析**：例如 pandas、NumPy\n'
 '- **人工智慧 / 機器學習**：例如 TensorFlow、PyTorch\n'
 '- **自動化工作**：例如批次改檔名、整理資料\n'
 '- **爬蟲**：抓取網頁資料\n'
 '\n'
 '如果你想，我也可以進一步用**超簡')


In [8]:
pp(response)

('Python 是一種**程式語言**，用來寫電腦程式、網站、資料分析、人工智慧、自動化腳本等。\n'
 '\n'
 '### 簡單來說\n'
 '你可以把 Python 想成是「跟電腦溝通的語言」。  \n'
 '你用 Python 寫出指令，電腦就照著做。\n'
 '\n'
 '### Python 的特色\n'
 '- **語法簡單、好讀**：很適合初學者\n'
 '- **用途廣泛**：可以做網站、資料處理、機器學習、爬蟲、自動化\n'
 '- **社群很大**：有很多現成套件可以用\n'
 '- **跨平台**：Windows、Mac、Linux 都能跑\n'
 '\n'
 '### 例子\n'
 '```python\n'
 'print("你好，Python！")\n'
 '```\n'
 '\n'
 '這段程式會在畫面上顯示：\n'
 '```python\n'
 '你好，Python！\n'
 '```\n'
 '\n'
 '### 常見用途\n'
 '- **網站開發**：例如 Django、Flask\n'
 '- **資料分析**：例如 pandas、NumPy\n'
 '- **人工智慧 / 機器學習**：例如 TensorFlow、PyTorch\n'
 '- **自動化工作**：例如批次改檔名、整理資料\n'
 '- **爬蟲**：抓取網頁資料\n'
 '\n'
 '如果你想，我也可以進一步用**超簡')


## API 可調參數 - Temperature 溫度

In [9]:
user_message = "鬆鬆軟軟的食物是? "

messages = [
    {
        "role": "user",
        "content": user_message
    }
]

response = get_openai_completion(messages, temperature=1)  # 可以改看看溫度
print(response)

「鬆鬆軟軟的食物」通常是指**口感蓬鬆、柔軟、容易咬**的食物，例如：

- **蛋糕**：戚風蛋糕、海綿蛋糕
- **麵包**：吐司、奶油麵包、菠蘿麵包
- **饅頭**：剛蒸好的白饅頭
- **蛋卷 / 蛋餅**
- **蒸蛋**
- **豆腐**：嫩豆腐、豆花
- **馬鈴薯泥**
- **舒芙蕾**
- **鬆餅**：口感軟軟蓬蓬的那種

如果你是想問「**適合小孩、老人、牙齒不好的人吃的鬆軟食物**」，我也可以幫你整理一份清單。


## API 可調參數 — 輸出可重現性

模型的輸出本質上仍是**非確定性的**。開發與測試時若每次都得到不同結果，會很難驗證行為是否正確。

### 本筆記的做法（Responses API）

上方的 `get_openai_completion` 走 **Responses API**，**沒有 `seed` 參數**。要提高輸出穩定性，請先固定：

- **model** 與 **prompt**（含 `instructions` / 對話角色）
- **temperature**（越低通常越穩定，`0` 最保守）

下方示範用 `temperature=0.8` 對同一問題多次執行，觀察仍可能出現的差異——這正是 LLM「難以完全控制」的起點。

### 延伸：舊版 Chat Completions 的 `seed`

從 1106 系列模型起，**Chat Completions** 曾提供 `seed`，搭配 `system_fingerprint` 可在非零 temperature 下盡可能重現輸出。那是舊 API 的控制介面；本課程新專案以 Responses API 為主，`seed` 不應再當成主線控制點。

* https://platform.openai.com/docs/guides/text-generation/reproducible-outputs（Chat Completions 脈絡）
* https://cookbook.openai.com/examples/deterministic_outputs_with_the_seed_parameter

In [16]:
user_message = "鬆鬆軟軟的食物是? "

messages = [
    {
        "role": "user",
        "content": user_message
    }
]

# Responses API 無 seed；固定 model + prompt，用 temperature 觀察輸出穩定性
response = get_openai_completion(messages, temperature=0.8)
print(response)

燉飯、燉湯、布丁、果凍等都是鬆鬆軟軟的食物。


## 使用 Instructions / System Message

Responses API 可以用 `instructions` 放全域行為約束，也可以在 `input` 陣列中保留 `system` / `user` / `assistant` 這類對話角色。教學上先沿用 role/content 陣列，讓學生理解訊息角色如何影響模型行為；進階 agent 或 production workflow 再把穩定規則提升到 `instructions`、工具設定或應用層 guardrails。

### 無 System Message

In [18]:
# 這是 completion 風格(蠻多教材仍這樣寫，包括 ChatGPT Prompt Engineering for Developers 課程)
user_message = """
請分類以下文字是 neutral, negative 或 positive
文字: 沙丘2超級好看! 讚
情緒:
"""

messages = [
    {
        "role": "user",
        "content": user_message
    }
]

response = get_openai_completion(messages, temperature=0.7)
print(response)


positive


### 有 System Message

In [20]:
# 可改成使用 system prompt 的風格: 把整體指示放在 system message
user_message = """
文字: 沙丘2超級好看! 讚
"""

messages = [
    {
        "role": "system",
        "content": "請分類以下文字是 neutral, negative 或 positive"
    },
    {
        "role": "user",
        "content": user_message
    }
]

response = get_openai_completion(messages, temperature=0.7)
print(response)

positive


## 連續對話 messages 的使用

In [22]:
# 第一輪問答
messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "2008年奧運在哪裡舉辦?"},
  ]

response1 = get_openai_completion(messages, temperature=0.3)
print(response1)



2008年奧運在中國的北京舉辦。


### 模型的幻覺現象 Hallucination

In [25]:
# 延續同一個對話的 第二輪問答
messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "2008年奧運在哪裡舉辦?"}, # 這是第一輪的 user 問句
      {"role": "assistant", "content": response1 }, # 這是第一輪的 AI 回覆
      {"role": "user", "content": "那2017年呢?"} # 這是第二輪的 user 問句
]

response2 = get_openai_completion(messages, temperature=0.3)
print(response2)


2017年世界大學運動會在臺灣的臺北舉辦，而非奧運。


In [26]:
# 延續同一個對話的 第二輪問答
messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "2008年奧運在哪裡舉辦?"}, # 這是第一輪的 user 問句
      {"role": "assistant", "content": response1 }, # 這是第一輪的 AI 回覆
      {"role": "user", "content": "那2017年呢? 如果沒舉辦，請回答沒舉辦"} # 這是第二輪的 user 問句
]

response2 = get_openai_completion(messages, temperature=0.3)
print(response2)



2017年並沒有舉辦夏季奧運會。


### 更多 Prompt 使用範例: https://platform.openai.com/examples



## Token 數的計算

除了 response 會告訴你實際使用 tokens，我們也可以自己先算

![image.png](attachment:image.png)

In [27]:
# !pip install tiktoken -q

In [10]:
import tiktoken

string = "哈囉哈囉"

encoding = tiktoken.encoding_for_model(OPENAI_MODEL)  # 現行 GPT-5.x / o 系列多用 o200k_base

num_tokens = len(encoding.encode(string))
print(num_tokens)



6


### Chat API 的 token 會再多一點

因為訊息格式（角色標記）的緣故，實際用量會比純內容多一些固定開銷。

參考: https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken

GPT-5.x / o 等 chat 模型與 completions 一樣以 token 計費，但因訊息式格式較難精算，故以上函式加入每則訊息的固定開銷。

In [ ]:
# messages = [
#   {"role": "system", "content": "You are a helpful, pattern-following assistant that translates corporate jargon into plain English."},
#   {"role": "system", "name":"example_user", "content": "New synergies will help drive top-line growth."},
#   {"role": "system", "name": "example_assistant", "content": "Things working well together will increase revenue."},
#   {"role": "system", "name":"example_user", "content": "Let's circle back when we have more bandwidth to touch base on opportunities for increased leverage."},
#   {"role": "system", "name": "example_assistant", "content": "Let's talk later when we're less busy about how to do better."},
#   {"role": "user", "content": "This late pivot means we don't have time to boil the ocean for the client deliverable."},
# ]

# model = OPENAI_MODEL

# print(f"{num_tokens_from_messages(messages, model)} prompt tokens counted.")
# # Should show ~126 total_tokens

In [30]:
import tiktoken

def num_tokens_from_messages(messages, model=OPENAI_MODEL):
    """回傳一組 chat messages 的 token 數（適用 GPT-5.x / o 系列）。"""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("o200k_base")  # 現行 chat 模型的預設編碼

    tokens_per_message = 3  # 每則訊息的格式開銷
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens -= 1
    num_tokens += 3  # 回覆會以 <assistant> 起始
    return num_tokens

In [33]:
# 使用範例
messages = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"}
]
print(num_tokens_from_messages(messages, model=OPENAI_MODEL))

17


## 故意超過 token 數造成錯誤

In [21]:
user_message = "故意撐爆長度" * 4000

messages = [
    {
        "role": "user",
        "content": user_message
    }
]

response = get_openai_completion(messages, temperature=0.7)
pp(response)


{'message': "This model's maximum context length is 16385 tokens. However, "
            'your messages resulted in 40007 tokens. Please reduce the length '
            'of the messages.',
 'type': 'invalid_request_error',
 'param': 'messages',
 'code': 'context_length_exceeded'}



## 官方錯誤代碼

使用 OpenAI API 時，了解各種可能出現的錯誤代碼是很重要的。這些錯誤代碼提供了在 API 請求過程中可能出現問題的線索以及如何解決問題的方法。

### 常見的 API 錯誤代碼

1. **身份驗證錯誤**：當 API 金鑰有問題時，會出現這些錯誤。
   - 401 Unauthorized：表示您的 API 金鑰缺失或無效。請確保您使用的是正確的 API 金鑰。

2. **請求錯誤**：這些錯誤是由於請求本身有問題。
   - 400 Bad Request：表示請求格式錯誤或無效。請仔細檢查請求參數。
   - 404 Not Foun`：表示找不到請求的資源。請確認端點和資源識別符。

3. **速率限制錯誤**：當您超過允許的請求數量時，會出現這些錯誤。
   - 429 Too Many Requests：表示您已達到速率限制。實施指數回退的重試機制來處理這種情況。

4. **伺服器錯誤**：這些錯誤表示伺服器端的問題。
   - 500 Internal Server Error：一般錯誤，表示伺服器出了問題。
   - 502 Bad Gateway：表示伺服器作為網關或代理時出現問題。
   - 503 Service Unavailable：表示伺服器目前無法處理請求，通常是由於暫時性過載或維護。

有關 OpenAI 錯誤代碼的更多詳細信息，請參閱[API 錯誤代碼解釋](https://help.openai.com/en/collections/3808446-api-error-codes-explained)和[Python 庫錯誤類型](https://platform.openai.com/docs/guides/error-codes/python-library-error-types)。

## 處理超時問題

大型請求處理時間可能較長，建議設定較長的 timeout 並實作指數回退重試。實際延遲與費率請參考官方定價頁（會變動）：https://openai.com/api/pricing/

流式回應也可以減少等待時間。流式傳輸在生成 token 時逐個發送回應，提供更即時的用戶體驗。

## 伺服器錯誤

相比 Azure OpenAI，目前 OpenAI 的穩定性稍差一點，有時會出現各種伺服器端錯誤。常見的伺服器錯誤包括：

- **伺服器在處理您的請求時發生錯誤。對此表示抱歉！您可以重試您的請求，或通過 help.openai.com 聯繫我們的幫助中心，如果您持續看到此錯誤。**
- **內部伺服器錯誤**
- **錯誤的網關**

遇到這些錯誤時，建議實施重試機制。[OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_handle_rate_limits.ipynb) 提供了一個基於 Python 的速率限制和重試解決方案，可以用來處理伺服器錯誤。

通過了解這些錯誤代碼並實施適當的處理策略，您可以創建更強大和更穩定


---

## 本章小結

1. **Responses API** 是 2026 新專案建議入口：用 `input` 傳入文字或 role/content 陣列，用 `output_text` 取得一般文字結果。
2. **Chat Completions 仍可相容舊系統**，但本課程新範例優先使用 `client.responses.create()`。
3. **raw HTTP vs SDK**：理解協定可看 requests；實務一律用 `openai>=2.0` 的 `OpenAI()` client（重試、型別、串流）。
4. **可控性槓桿**：temperature、instructions/input、max_output_tokens 都會影響輸出邊界。
5. **token 計算**用 `tiktoken.encoding_for_model()`；不確定模型時 fallback 到 `o200k_base`。
6. **錯誤處理**（429 速率限制、逾時、5xx）是部署可控管線的前提。